- Ce script automatise le téléchargement d'images satellite Sentinel-2 pour des points GPS spécifiques en Afrique pour l'année 2023, en utilisant l'API de Google Earth Engine. Tout d'abord, il installe les packages nécessaires, dont `earthengine-api`, `requests`, et `Pillow`, puis authentifie l'utilisateur et initialise l'API Earth Engine pour accéder aux données Sentinel-2.

- Le script spécifie un dossier de destination pour sauvegarder les images téléchargées et crée ce dossier s'il n'existe pas déjà. Il définit ensuite une fonction `download_sentinel_image`, qui prend en entrée la latitude et la longitude d'un point. Cette fonction vérifie d'abord si l'image existe déjà dans le dossier pour éviter les téléchargements redondants. Elle utilise ensuite Earth Engine pour rechercher des images Sentinel-2 autour du point GPS spécifié, en filtrant les images en fonction de la couverture nuageuse et de la date (2023 et 2022 en cas de besoin). Elle prend la médiane des images sélectionnées pour obtenir une image représentative sans nuages.

- Pour chaque point GPS, le script télécharge une image avec les bandes rouges, vertes, et bleues (B4, B3, B2) dans un format JPEG avec une résolution de 10 mètres. Si une connexion échoue, la fonction tente de nouveau jusqu'à trois fois. Finalement, le script charge un fichier CSV contenant les coordonnées des points d'intérêt et itère sur chaque ligne pour appeler la fonction de téléchargement, affichant une barre de progression pour suivre l’avancement.

In [ ]:
# Installer les packages nécessaires (à exécuter uniquement si non installés)
!pip install earthengine-api pillow requests tqdm

import ee
import requests
from PIL import Image
import os
import time
import pandas as pd
from io import BytesIO
from tqdm import tqdm

In [2]:
# Authentification et initialisation de Earth Engine
ee.Authenticate()
ee.Initialize()

In [3]:
# Paramètres
folder_path = "D:\\wealth_predict_sentinel\\Data\\downloaded\\Image_satellite_Divers_Pays_Afrique_Zoom_14_Sentinel_2_an_2023"
image_format = "jpeg"

# Création du dossier si non existant
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

In [4]:
# Modifier la fonction pour inclure uniquement les bandes spécifiées
def download_sentinel_image(latitude, longitude, folder_path, image_format="jpeg"):
    # Nom du fichier basé sur les coordonnées et la résolution
    file_name = f"lat_{latitude}_lon_{longitude}_Zoom_14.{image_format}"
    file_path = os.path.join(folder_path, file_name)

    # Vérifie si le fichier existe déjà
    if os.path.exists(file_path):
        return False  # Ne télécharge pas si l'image est déjà présente

    point = ee.Geometry.Point([longitude, latitude])
    date_ranges = [('2023-01-01', '2023-12-31', [10, 20, 30]), ('2022-01-01', '2022-12-31', [10, 20, 30])]
    image = None

    for start_date, end_date, cloud_coverage_limits in date_ranges:
        for cloud_coverage in cloud_coverage_limits:
            image_collection = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
                .filterBounds(point) \
                .filterDate(start_date, end_date) \
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_coverage)) \
                .select(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12']) \
                .sort('CLOUDY_PIXEL_PERCENTAGE')
            
            image = image_collection.median().reproject('EPSG:3857', None, 10).clip(point.buffer(2000).bounds())
            
            if image.reduceRegion(ee.Reducer.count(), point.buffer(2000).bounds(), scale=10).getInfo():
                break
        if image.reduceRegion(ee.Reducer.count(), point.buffer(2000).bounds(), scale=10).getInfo():
            break

    if image is None:
        print(f"Aucune image trouvée pour ({latitude}, {longitude}) dans les conditions spécifiées.")
        return False

    # Paramètres de visualisation et de téléchargement
    region = point.buffer(2000).bounds()
    vis_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 6000}
    thumb_params = {
        'region': region.getInfo(),
        'dimensions': '400x400',
        'format': 'png',
        'crs': 'EPSG:4326'
    }
    url = image.visualize(**vis_params).getThumbURL(thumb_params)

    for attempt in range(3):
        try:
            response = requests.get(url, timeout=30)
            if response.status_code == 200:
                img = Image.open(BytesIO(response.content))
                img.convert("RGB").save(file_path, image_format.upper())
                return True
            else:
                print(f"Erreur lors du téléchargement ({latitude}, {longitude}): {response.status_code}")
                return False
        except requests.exceptions.RequestException:
            print(f"Échec de connexion pour ({latitude}, {longitude}), tentative dans 1 minute.")
            time.sleep(60)

    print(f"Échec définitif pour ({latitude}, {longitude}) après plusieurs tentatives.")
    return False


In [ ]:
# Charger le fichier CSV
csv_path = r"D:\wealth_predict_sentinel\Data\processed_csv\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal_classified_8k_obs.csv"
df = pd.read_csv(csv_path)

# Boucle de téléchargement avec barre de progression
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Téléchargement des images"):
    latitude = row['latitude']
    longitude = row['longitude']
    
    # Télécharger l'image pour chaque point
    download_sentinel_image(latitude, longitude, folder_path, image_format)